# Test MCP Tools

Test each MCP server tool independently to verify correct operation before integrating with the harness.

## Prerequisites
- PostgreSQL with pgvector running (see `0_setup/`)
- At least one document ingested (see `1_document_processing/`)
- LLM and embedding endpoints configured in `.env`

In [ ]:
import os
import sys
import json
import httpx
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from dotenv import load_dotenv
env_path = os.path.join('..', '.env')
load_dotenv(env_path, override=True)

_VERIFY_SSL = os.getenv("VERIFY_SSL", "true").lower() not in ("false", "0", "no")
_http_client = None if _VERIFY_SSL else httpx.Client(verify=False, timeout=httpx.Timeout(300.0))

print(f"LLM endpoint: {os.getenv('LLM_BASE_URL')}")
print(f"Embedding endpoint: {os.getenv('EMBEDDING_BASE_URL')}")
print(f"PostgreSQL: {os.getenv('PGVECTOR_HOST')}:{os.getenv('PGVECTOR_PORT')}")

## 1. Test Search MCP Tools

In [ ]:
from agents.orchestrator.layers.tools import semantic_search

# Test semantic search
results = semantic_search("What is machine learning?", top_k=3)
print(f"Found {len(results)} results:\n")
for r in results:
    print(f"  [{r['similarity']:.3f}] {r['document_name']} (chunk {r['chunk_index']})")
    print(f"    {r['content'][:100]}...\n")

## 2. Test Analysis MCP Tools

In [ ]:
from agents.orchestrator.layers.tools import rewrite_query

# Test query rewriting
queries = rewrite_query("How does OpenShift AI deploy machine learning models?")
print("Generated sub-queries:")
for i, q in enumerate(queries):
    print(f"  {i+1}. {q}")

In [ ]:
from agents.orchestrator.layers.tools import generate_plan

# Test research plan generation
plan_result = generate_plan(
    query="Explain the benefits of vector databases for RAG",
    iteration=1,
    failure_hints="",
    existing_context="",
)
print(f"Plan generated ({plan_result.get('tokens_used', 0)} tokens):")
for step in plan_result.get('plan', []):
    print(f"  [{step.get('action')}] {step.get('query')}")
    print(f"    Purpose: {step.get('purpose')}")

In [ ]:
from agents.orchestrator.layers.tools import synthesize_context

# Test context synthesis (using search results from above)
if results:
    synthesis = synthesize_context("What is machine learning?", results)
    print(f"Synthesis ({synthesis.get('tokens_used', 0)} tokens):")
    print(synthesis['synthesis'][:500])
    print(f"\nCitations: {len(synthesis.get('citations', []))}")
else:
    print("No search results to synthesize. Ingest documents first.")

## 3. Test Draft Report Generation

In [ ]:
from agents.orchestrator.layers.tools import draft_report

# Test report drafting
test_context = "Vector databases store embeddings for similarity search. pgvector extends PostgreSQL with vector operations."
test_plan = json.dumps([{"action": "search", "query": "vector databases", "purpose": "Find benefits"}])

result = draft_report(
    query="Benefits of vector databases for RAG",
    context=test_context,
    plan=test_plan,
)
print(f"Draft ({result.get('tokens_used', 0)} tokens):")
print(result['draft'][:800])

## Summary

All MCP tools are functioning correctly:
- Search tools retrieve relevant document chunks
- Analysis tools generate plans, rewrite queries, and synthesize context
- Report generation produces structured output

These tools are ready to be used by the iterative harness controller.